# User Tower
### Idea: turn each user into one dense vector that summarizes their anime preferences  


inputs to user tower for each user:
- user_idx (encoded user ID)
- history_ids (This is a fixed-length list of the user's most important anime history)
- history_weights (This is one weight per anime in history_ids)  
- numeric_feats (user's numeric behavior summary)  

        avg_rating_nonzero,
        completion_ratio,
        dropped_ratio,
        avg_progress_ratio,
        fraction_long_shows,
        log_num_interactions,
        fraction_rated


these four will be input into the user tower and the user tower will output a user embedding vector to represent the user  

improvement for consideration: adding genre preference

In [1]:
import os
import math
import random
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Configuration

In [3]:
from google.colab import drive
drive.mount('/content/drive')


ANIMELIST_PATH = "/content/drive/MyDrive/bt4222_anime_datasets/animelist.csv"
ANIME_PATH = "/content/drive/MyDrive/bt4222_anime_datasets/anime.csv"

CHUNK_SIZE = 500_000          # can increase later
MAX_HISTORY_LEN = 20          # can increase later

MIN_USER_INTERACTIONS = 10    # can increase later
MIN_ITEM_INTERACTIONS = 10    # can increase later

BATCH_SIZE = 256
USER_ID_EMB_DIM = 64
ANIME_EMB_DIM = 64
OUTPUT_DIM = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


STATUS_COMPLETED = 2
STATUS_WATCHING = 1
STATUS_ON_HOLD = 3
STATUS_DROPPED = 4
STATUS_PLAN_TO_WATCH = 6

Mounted at /content/drive


# Utility functions

In [4]:
def pad_or_truncate(seq, max_len, pad_value=0):
    seq = seq[:max_len]
    if len(seq) < max_len:
        seq = seq + [pad_value] * (max_len - len(seq))
    return seq


def normalize_episode_count(ep):
    if pd.isna(ep):
        return 0
    if isinstance(ep, str):
        ep = ep.strip()
        if ep == "" or ep.lower() == "unknown":
            return 0
    try:
        ep = int(float(ep))
        return max(ep, 0)
    except:
        return 0

# Load anime metadata

In [5]:
print("Loading anime metadata...")

anime_df = (
    pd.read_csv(ANIME_PATH, usecols=["MAL_ID", "Episodes"])
      .rename(columns={"MAL_ID": "anime_id", "Episodes": "episodes"})
)

anime_df["anime_id"] = pd.to_numeric(anime_df["anime_id"], errors="coerce")
anime_df = anime_df.dropna(subset=["anime_id"]).copy()
anime_df["anime_id"] = anime_df["anime_id"].astype(int)
anime_df["episodes"] = anime_df["episodes"].apply(normalize_episode_count)

anime_episode_map = dict(zip(anime_df["anime_id"], anime_df["episodes"]))

print("anime.csv shape:", anime_df.shape)
print("anime_episode_map size:", len(anime_episode_map))


Loading anime metadata...
anime.csv shape: (17562, 2)
anime_episode_map size: 17562


# First pass over animelist.csv
# Count user/item interactions for filtering

In [6]:
print("\nFirst pass: counting user/item interactions...")

usecols = ["user_id", "anime_id", "rating", "watching_status", "watched_episodes"]

user_counter = Counter()
item_counter = Counter()

for chunk in pd.read_csv(ANIMELIST_PATH, usecols=usecols, chunksize=CHUNK_SIZE):
    chunk = chunk.dropna(subset=["user_id", "anime_id"]).copy()

    chunk["user_id"] = pd.to_numeric(chunk["user_id"], errors="coerce")
    chunk["anime_id"] = pd.to_numeric(chunk["anime_id"], errors="coerce")
    chunk = chunk.dropna(subset=["user_id", "anime_id"]).copy()

    chunk["user_id"] = chunk["user_id"].astype(int)
    chunk["anime_id"] = chunk["anime_id"].astype(int)

    user_counter.update(chunk["user_id"].value_counts().to_dict())
    item_counter.update(chunk["anime_id"].value_counts().to_dict())

valid_users = {u for u, c in user_counter.items() if c >= MIN_USER_INTERACTIONS}
valid_items = {a for a, c in item_counter.items() if c >= MIN_ITEM_INTERACTIONS}

print("Users before filtering:", len(user_counter))
print("Items before filtering:", len(item_counter))
print("Valid users:", len(valid_users))
print("Valid items:", len(valid_items))


First pass: counting user/item interactions...
Users before filtering: 325770
Items before filtering: 17562
Valid users: 311238
Valid items: 17550


# Encode user/item IDs

In [7]:
print("\nEncoding IDs...")

sorted_valid_users = sorted(valid_users)
sorted_valid_items = sorted(valid_items)

user2idx = {u: i + 1 for i, u in enumerate(sorted_valid_users)}
anime2idx = {a: i + 1 for i, a in enumerate(sorted_valid_items)}

idx2user = {i: u for u, i in user2idx.items()}
idx2anime = {i: a for a, i in anime2idx.items()}

NUM_USERS = len(user2idx) + 1
NUM_ANIME = len(anime2idx) + 1

print("NUM_USERS:", NUM_USERS)
print("NUM_ANIME:", NUM_ANIME)


Encoding IDs...
NUM_USERS: 311239
NUM_ANIME: 17551


# Second pass over animelist.csv
# Build user histories + important numeric features (~20 min to run using CPU)

In [8]:
print("\nSecond pass: building user features...")

user_stats = defaultdict(lambda: {
    "num_interactions": 0,
    "num_rated": 0,
    "rating_sum": 0.0,
    "num_completed": 0,
    "num_dropped": 0,
    "progress_ratio_sum": 0.0,
    "progress_ratio_count": 0,
    "num_long_shows": 0,
    "num_known_episode_shows": 0,
})

# only keep a bounded number of strong candidates per user
MAX_CANDIDATES_PER_USER = MAX_HISTORY_LEN * 3
user_history_candidates = defaultdict(list)

for chunk in pd.read_csv(ANIMELIST_PATH, usecols=usecols, chunksize=CHUNK_SIZE):
    chunk = chunk.dropna(subset=["user_id", "anime_id"]).copy()

    chunk["user_id"] = pd.to_numeric(chunk["user_id"], errors="coerce")
    chunk["anime_id"] = pd.to_numeric(chunk["anime_id"], errors="coerce")
    chunk["rating"] = pd.to_numeric(chunk["rating"], errors="coerce").fillna(0)
    chunk["watching_status"] = pd.to_numeric(chunk["watching_status"], errors="coerce").fillna(0)
    chunk["watched_episodes"] = pd.to_numeric(chunk["watched_episodes"], errors="coerce").fillna(0)

    chunk = chunk.dropna(subset=["user_id", "anime_id"]).copy()

    chunk["user_id"] = chunk["user_id"].astype(int)
    chunk["anime_id"] = chunk["anime_id"].astype(int)
    chunk["rating"] = chunk["rating"].astype(float)
    chunk["watching_status"] = chunk["watching_status"].astype(int)
    chunk["watched_episodes"] = chunk["watched_episodes"].astype(float)

    chunk = chunk[
        chunk["user_id"].isin(valid_users) &
        chunk["anime_id"].isin(valid_items)
    ].copy()

    if len(chunk) == 0:
        continue

    for row in chunk.itertuples(index=False):
        user_id = row.user_id
        anime_id = row.anime_id
        rating = float(row.rating)
        status = int(row.watching_status)
        watched_eps = max(float(row.watched_episodes), 0.0)

        user_idx = user2idx[user_id]
        anime_idx = anime2idx[anime_id]

        stats = user_stats[user_idx]
        stats["num_interactions"] += 1

        if rating > 0:
            stats["num_rated"] += 1
            stats["rating_sum"] += rating

        if status == STATUS_COMPLETED:
            stats["num_completed"] += 1
        elif status == STATUS_DROPPED:
            stats["num_dropped"] += 1

        total_eps = anime_episode_map.get(anime_id, 0)

        if total_eps > 0:
            stats["num_known_episode_shows"] += 1
            progress_ratio = min(watched_eps / total_eps, 1.0)
            stats["progress_ratio_sum"] += progress_ratio
            stats["progress_ratio_count"] += 1

            if total_eps >= 24:
                stats["num_long_shows"] += 1
        else:
            progress_ratio = min(watched_eps / 12.0, 1.0) if watched_eps > 0 else 0.1

        if rating > 0:
            rating_weight = rating / 10.0
        else:
            rating_weight = 0.3

        if status == STATUS_COMPLETED:
            status_weight = 1.0
        elif status == STATUS_DROPPED:
            status_weight = 0.1
        elif status == STATUS_WATCHING:
            status_weight = 0.7
        elif status == STATUS_ON_HOLD:
            status_weight = 0.4
        elif status == STATUS_PLAN_TO_WATCH:
            status_weight = 0.2
        else:
            status_weight = 0.2

        final_weight = 0.5 * rating_weight + 0.3 * status_weight + 0.2 * progress_ratio
        sort_score = (rating if rating > 0 else 0) * 10 + status_weight * 5 + progress_ratio


        # periodically trim to prevent RAM blow-up
        #####################################################################
        lst = user_history_candidates[user_idx]
        lst.append((anime_idx, final_weight, sort_score))


        if len(lst) > MAX_CANDIDATES_PER_USER:
            lst.sort(key=lambda x: x[2], reverse=True)
            user_history_candidates[user_idx] = lst[:MAX_CANDIDATES_PER_USER]
        #####################################################################

        # if RAM not a concern, use this line below
        # user_history_candidates[user_idx].append((anime_idx, final_weight, sort_score))

print("Finished raw feature aggregation.")


Second pass: building user features...
Finished raw feature aggregation.


# Finalize per-user inputs

In [9]:
print("\nFinalizing user histories and numeric features...")

user_history_ids = {}
user_history_weights = {}
user_numeric_features = {}

for user_idx, stats in user_stats.items():
    n = stats["num_interactions"]
    if n == 0:
        continue

    num_rated = stats["num_rated"]

    avg_rating_nonzero = stats["rating_sum"] / num_rated if num_rated > 0 else 0.0
    completion_ratio = stats["num_completed"] / n
    dropped_ratio = stats["num_dropped"] / n
    avg_progress_ratio = (
        stats["progress_ratio_sum"] / stats["progress_ratio_count"]
        if stats["progress_ratio_count"] > 0 else 0.0
    )
    fraction_long_shows = (
        stats["num_long_shows"] / stats["num_known_episode_shows"]
        if stats["num_known_episode_shows"] > 0 else 0.0
    )
    log_num_interactions = math.log1p(n)
    fraction_rated = num_rated / n

    numeric_feats = np.array([
        avg_rating_nonzero,
        completion_ratio,
        dropped_ratio,
        avg_progress_ratio,
        fraction_long_shows,
        log_num_interactions,
        fraction_rated,
    ], dtype=np.float32)

    user_numeric_features[user_idx] = numeric_feats

    candidates = sorted(user_history_candidates[user_idx], key=lambda x: x[2], reverse=True)

    seen = set()
    deduped = []
    for anime_idx, weight, sort_score in candidates:
        if anime_idx not in seen:
            deduped.append((anime_idx, weight))
            seen.add(anime_idx)

    deduped = deduped[:MAX_HISTORY_LEN]

    history_ids = [x[0] for x in deduped]
    history_weights = [float(x[1]) for x in deduped]

    history_ids = pad_or_truncate(history_ids, MAX_HISTORY_LEN, pad_value=0)
    history_weights = pad_or_truncate(history_weights, MAX_HISTORY_LEN, pad_value=0.0)

    user_history_ids[user_idx] = np.array(history_ids, dtype=np.int64)
    user_history_weights[user_idx] = np.array(history_weights, dtype=np.float32)

NUMERIC_FEAT_DIM = 7
all_user_indices = sorted(user_numeric_features.keys())

print("Users with finalized features:", len(all_user_indices))
print("NUMERIC_FEAT_DIM:", NUMERIC_FEAT_DIM)


Finalizing user histories and numeric features...
Users with finalized features: 311238
NUMERIC_FEAT_DIM: 7


# Normalize numeric features

In [10]:
print("\nNormalizing numeric features...")

numeric_matrix = np.stack([user_numeric_features[u] for u in all_user_indices], axis=0)
feat_mean = numeric_matrix.mean(axis=0)
feat_std = numeric_matrix.std(axis=0)
feat_std = np.where(feat_std < 1e-8, 1.0, feat_std)

for u in all_user_indices:
    user_numeric_features[u] = ((user_numeric_features[u] - feat_mean) / feat_std).astype(np.float32)


Normalizing numeric features...


# Dataset

In [ ]:
class ImportantFeatureUserTowerDataset(Dataset):
    def __init__(self, user_indices, user_history_ids, user_history_weights, user_numeric_features):
        self.user_indices = user_indices
        self.user_history_ids = user_history_ids
        self.user_history_weights = user_history_weights
        self.user_numeric_features = user_numeric_features

    def __len__(self):
        return len(self.user_indices)

    def __getitem__(self, idx):
        user_idx = self.user_indices[idx]
        return {
            "user_idx": torch.tensor(user_idx, dtype=torch.long),
            "history_ids": torch.tensor(self.user_history_ids[user_idx], dtype=torch.long),
            "history_weights": torch.tensor(self.user_history_weights[user_idx], dtype=torch.float32),
            "numeric_feats": torch.tensor(self.user_numeric_features[user_idx], dtype=torch.float32),
        }


dataset = ImportantFeatureUserTowerDataset(
    all_user_indices,
    user_history_ids,
    user_history_weights,
    user_numeric_features
)

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print("Dataset size:", len(dataset))


# User Tower

In [ ]:
class ImportantFeatureUserTower(nn.Module):
    def __init__(
        self,
        num_users,
        num_anime,
        numeric_feat_dim,
        user_id_emb_dim=64,
        anime_emb_dim=64,
        hidden_dims=(256, 128),
        output_dim=64,
        padding_idx=0,
        dropout=0.2
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_embeddings=num_users,
            embedding_dim=user_id_emb_dim,
            padding_idx=padding_idx
        )

        self.history_anime_embedding = nn.Embedding(
            num_embeddings=num_anime,
            embedding_dim=anime_emb_dim,
            padding_idx=padding_idx
        )

        self.numeric_mlp = nn.Sequential(
            nn.Linear(numeric_feat_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 64),
            nn.ReLU(),
        )

        fusion_input_dim = user_id_emb_dim + anime_emb_dim + 64

        self.fusion_mlp = nn.Sequential(
            nn.Linear(fusion_input_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[1], output_dim)
        )

    def weighted_mean_pool(self, seq_emb, weights, mask):
        weights = weights * mask.float()
        weights_sum = weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
        norm_weights = weights / weights_sum
        pooled = torch.sum(seq_emb * norm_weights.unsqueeze(-1), dim=1)
        return pooled

    def forward(self, user_ids, history_anime_ids, history_weights, user_numeric_feats):
        user_id_emb = self.user_embedding(user_ids)
        history_emb = self.history_anime_embedding(history_anime_ids)
        mask = (history_anime_ids != 0)
        history_pooled = self.weighted_mean_pool(history_emb, history_weights, mask)
        numeric_vec = self.numeric_mlp(user_numeric_feats)

        x = torch.cat([user_id_emb, history_pooled, numeric_vec], dim=1)
        user_vec = self.fusion_mlp(x)
        user_vec = F.normalize(user_vec, p=2, dim=1)
        return user_vec


model = ImportantFeatureUserTower(
    num_users=NUM_USERS,
    num_anime=NUM_ANIME,
    numeric_feat_dim=NUMERIC_FEAT_DIM,
    user_id_emb_dim=USER_ID_EMB_DIM,
    anime_emb_dim=ANIME_EMB_DIM,
    hidden_dims=(256, 128),
    output_dim=OUTPUT_DIM,
    dropout=0.2
).to(DEVICE)

print(model)

# Test one batch

In [ ]:
batch = next(iter(loader))

user_idx = batch["user_idx"].to(DEVICE)
history_ids = batch["history_ids"].to(DEVICE)
history_weights = batch["history_weights"].to(DEVICE)
numeric_feats = batch["numeric_feats"].to(DEVICE)

with torch.no_grad():
    user_vec = model(
        user_ids=user_idx,
        history_anime_ids=history_ids,
        history_weights=history_weights,
        user_numeric_feats=numeric_feats
    )

print("\nUser embedding shape:", user_vec.shape)
print(user_vec[:2])

# Encode all users function

In [ ]:
def encode_all_users(model, loader, device="cpu"):
    model.eval()
    all_user_ids = []
    all_user_vecs = []

    with torch.no_grad():
        for batch in loader:
            user_idx = batch["user_idx"].to(device)
            history_ids = batch["history_ids"].to(device)
            history_weights = batch["history_weights"].to(device)
            numeric_feats = batch["numeric_feats"].to(device)

            vecs = model(
                user_ids=user_idx,
                history_anime_ids=history_ids,
                history_weights=history_weights,
                user_numeric_feats=numeric_feats
            )

            all_user_ids.extend(user_idx.cpu().numpy().tolist())
            all_user_vecs.append(vecs.cpu())

    all_user_vecs = torch.cat(all_user_vecs, dim=0)
    return all_user_ids, all_user_vecs

# Encode all users

In [ ]:
# user_ids_encoded, user_embeddings = encode_all_users(model, loader, device=DEVICE)
# print(user_embeddings.shape)